[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/03_autogen.ipynb)

# Part 3 — Conversational multi-agent (AutoGen)

> **Control flow: emergent, from the conversation.** Who speaks next determines what happens next.

The pressure that produces this pattern: genuinely **different expertise** needs different system
prompts, different tools, and different incentives — and you want the disagreement on the record.

In [ ]:
# Setup -- same as 00_api_access.ipynb
import os, sys, time, json, textwrap
if 'google.colab' in sys.modules:
    %pip install -U -q google-genai

from google import genai
from google.genai import types as gtypes

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def generate_with_retry(*, contents, config=None, max_attempts=6):
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)


def llm_text(system_prompt, user_prompt, temperature=0.2):
    resp = generate_with_retry(
        contents=user_prompt,
        config=gtypes.GenerateContentConfig(
            system_instruction=system_prompt, temperature=temperature))
    return resp.text or ""

print(f"Gemini client ready (model={MODEL}).")

### 3.1 The syntax

```python
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination

modeler = AssistantAgent("modeler", model_client=client,
    system_message="You propose discretizations. Be concrete and specific.")

analyst = AssistantAgent("analyst", model_client=client,
    system_message="""You are a numerical analyst. Challenge the modeler's proposal on
    stability, conditioning, and convergence rate. Do not be agreeable; your value is
    in the objection.""")

critic = AssistantAgent("critic", model_client=client,
    system_message="Judge the exchange. Say APPROVE only when genuinely convinced.")

team = RoundRobinGroupChat([modeler, analyst, critic],
    termination_condition=TextMentionTermination("APPROVE"))

await team.run(task="Choose a discretization for advection-dominated flow.")
```

Note `autogen_agentchat` (v0.4+), not the older `autogen`/`pyautogen` packages — the API changed
substantially and most tutorials you will find online are for the old one.

In [ ]:
try:
    import autogen_agentchat
    HAVE_AUTOGEN = True
    print("autogen_agentchat is available.")
except ImportError:
    HAVE_AUTOGEN = False
    print("autogen_agentchat not installed — using the from-scratch version below.")

### 3.2 The same thing in 15 lines

A round-robin group chat is a list of system prompts, a shared transcript, and a stopping rule.

In [ ]:
def group_chat(agents, task, max_rounds=3, terminate_on="APPROVE", verbose=True):
    """agents: list of (name, system_prompt). Returns the transcript."""
    transcript = [("user", task)]
    for rnd in range(max_rounds):
        for name, system in agents:
            history = "\n\n".join(f"[{who}] {what}" for who, what in transcript)
            reply = llm_text(system, f"Conversation so far:\n\n{history}\n\nYour turn, {name}.")
            reply = reply.strip()
            transcript.append((name, reply))
            if verbose:
                print(f"\n--- [{name}] " + "-" * 50)
                print(textwrap.fill(reply, 92)[:900])
            if terminate_on in reply:
                return transcript
    return transcript

### 3.3 The exemplar: a design review

The toy task does not work here, and that is informative — there is nothing to argue about in
`13 * 47 + 8`. A conversation pattern needs a question with genuine tension in it.

So: **choosing a discretization for advection-dominated flow.** No single right answer, real
tradeoffs, and the *reasoning* is the deliverable.

In [ ]:
AGENTS = [
    ("modeler",
     "You propose discretizations for PDE problems. Be concrete and specific: name the method "
     "and the parameters. Two or three sentences. Respond to objections rather than repeating "
     "yourself."),
    ("analyst",
     "You are a numerical analyst. Challenge the modeler's proposal on stability, conditioning, "
     "and convergence rate. Be specific about the failure mode you are worried about. Do not be "
     "agreeable -- your value to this conversation is the objection. Two or three sentences."),
    ("critic",
     "You judge the exchange between a modeler and a numerical analyst. If the analyst's "
     "objection has been genuinely answered, reply with exactly APPROVE followed by one sentence "
     "of justification. Otherwise state in one sentence what is still unresolved. Do not say "
     "APPROVE merely because the discussion is polite or has gone on a while."),
]

TASK = ("Choose a spatial discretization for steady advection-diffusion at Peclet number ~500 "
        "on an unstructured triangular mesh. State the method and any stabilization parameter.")

tr = group_chat(AGENTS, TASK, max_rounds=2)
print(f"\n\n=== {len(tr)-1} replies; terminated: {'APPROVE' in tr[-1][1]}")

### 3.4 When to reach for conversation — and the honest caveat

**Use it when** disagreement is the product: the roles have different tools and different
incentives, and you want an auditable argument rather than a confident paragraph.

**The caveat, stated plainly:** this is the easiest of the four patterns to fool yourself with.

- Three LLMs agreeing is **not** three experts agreeing. They share a prior, a training corpus,
  and a tendency toward agreeableness. A "critic" that approves everything has told you nothing.
- Distinct **tools** per role helps far more than distinct adjectives in the system prompt. An
  analyst who can actually *run* a stability calculation is a different thing from one instructed
  to "be skeptical."
- Watch the run above for the critic approving too early. If it does, that is the pattern's
  characteristic failure — not a bug in the prompt.

**Cheapest useful version:** two agents, one of which has a tool the other lacks.